In [16]:
import pandas as pd
from enum import Enum
from pydantic import BaseModel
from typing import Optional
import json

In [24]:
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/moises/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [25]:
from nltk.tokenize import PunktTokenizer

sent_detector = PunktTokenizer("portuguese")

def sentencize_df(df, text_col, id_col=None):
    rows = []
    for i, row in df.iterrows():
        text = str(row[text_col])
        text = "\n".join([line.strip() for line in text.split("\n") if line.strip()])
        sentences = sent_detector.tokenize(text)
        for j, sent in enumerate(sentences):
            entry = {
                "article_idx": i,
                "sentence_idx": j,
                "sentence": sent,
            }
            if id_col:
                entry[id_col] = row[id_col]
            rows.append(entry)
    return pd.DataFrame(rows)


In [5]:
portuguese_df = pd.read_csv("FakeTrueBr_corpus.csv")
english_df = pd.read_csv("task3-df.csv")

In [17]:
portuguese_df.columns

Index(['title_fake', 'fake', 'link_f', 'true', 'link_t'], dtype='str')

In [20]:
portuguese_df["fake"].iloc[0]

'carnaval em olinda. arrastão monstro. fazuele isso foi só um aperitivo do que tá por vir no carnaval país comandado por bandidos, dá nisso. , 172, sexta feira de carnaval, em olinda os pobres meninos do luladrão promovendo um arrastão, nas ruas de olinda, atacando turistas para roubar um ou outro celular e ter um dinheirinho prá tomar uma cervejinha parecem mais uma praga de gafanhotos atacando uma plantação! o brasil está entregue à criminalidade com a proteção do presidente!  o amor venceu '

In [7]:
len(portuguese_df)

1791

In [10]:
english_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12291 entries, 0 to 12290
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   article_id  12291 non-null  int64
 1   text_id     12291 non-null  int64
 2   label       4713 non-null   str  
 3   text        12291 non-null  str  
dtypes: int64(2), str(2)
memory usage: 384.2 KB


In [12]:
english_df_cleaned = english_df[english_df['label'].notna()]


In [15]:
(
    english_df_cleaned['label']
    .dropna()
    .str.split(',')      # "A,B" → ["A", "B"]
    .explode()           # uma linha por label
    .str.strip()         # remove espaços acidentais
    .value_counts()
)

label
Loaded_Language                    2210
Name_Calling-Labeling              1184
Doubt                               679
Repetition                          657
Exaggeration-Minimisation           554
Appeal_to_Fear-Prejudice            433
Flag_Waving                         376
Causal_Oversimplification           227
False_Dilemma-No_Choice             182
Appeal_to_Authority                 174
Slogans                             173
Conversation_Killer                 113
Red_Herring                          61
Guilt_by_Association                 59
Appeal_to_Popularity                 47
Appeal_to_Hypocrisy                  46
Obfuscation-Vagueness-Confusion      31
Straw_Man                            22
Whataboutism                         17
Name: count, dtype: int64

In [ ]:
PROPAGANDA_LABELS = [
    "Loaded_Language",
    "Name_Calling-Labeling",
    "Doubt",
    "Repetition",
    "Exaggeration-Minimisation",
    "Appeal_to_Fear-Prejudice",
    "Flag_Waving",
    "Causal_Oversimplification",
    "False_Dilemma-No_Choice",
    "Appeal_to_Authority",
    "Slogans",
    "Conversation_Killer",
    "Red_Herring",
    "Guilt_by_Association",
    "Appeal_to_Popularity",
    "Appeal_to_Hypocrisy",
    "Obfuscation-Vagueness-Confusion",
    "Straw_Man",
    "Whataboutism",
    "None",
]

In [ ]:
SYSTEM_PROMPT = """You are an expert in political propaganda and persuasion techniques.

Your task: given a text in Portuguese, identify which propaganda techniques are used.
You must return a JSON with the field "labels" containing a list of techniques from:

{labels}

Definitions:
- Appeal_to_Authority: stating that a claim is true simply because an authority or expert said so, without other supporting evidence
- Appeal_to_Popularity: claiming something is true or good because many people believe or do it (bandwagon fallacy)
- Appeal_to_Fear-Prejudice: building support by instilling anxiety or panic toward an alternative, often with exaggerated warnings
- Flag_Waving: playing on strong national or group feeling to justify or promote an action or idea
- Causal_Oversimplification: assuming a single cause when there are multiple causes for an issue
- False_Dilemma-No_Choice: presenting only two options while ignoring other alternatives
- Straw_Man: misrepresenting someone's argument to make it easier to attack
- Red_Herring: introducing irrelevant material to divert attention away from the main issue
- Whataboutism: deflecting criticism by pointing to the opponent's similar faults
- Slogans: brief, striking phrases that act as emotional shortcuts, often including stereotyping
- Conversation_Killer: phrases that shut down debate by asserting the issue is too obvious or settled to discuss
- Loaded_Language: words or phrases with strong emotional implications used to influence the audience
- Repetition: repeating the same message so the audience gradually accepts it
- Exaggeration-Minimisation: representing something in an excessive manner or making something seem less important than it is
- Obfuscation-Vagueness-Confusion: using confusing or vague language to blur the point
- Name_Calling-Labeling: attaching a label to a person or entity to intimidate or associate them with something positive or negative
- Doubt: questioning the credibility of someone or something without evidence
- Guilt_by_Association: discrediting a person by associating them with a negatively viewed group or idea
- Appeal_to_Hypocrisy: discrediting an argument by claiming the person making it is guilty of the same thing (tu quoque)
- None: no propaganda technique detected

Return only JSON: {{"labels": [...], "justification": "..."}}
""".format(labels=", ".join(PROPAGANDA_LABELS))

In [21]:
class LLMLabeler:
    def __init__(self, model: str = "qwen2.5:7b", backend: str = "ollama",
                 few_shot_df: Optional[pd.DataFrame] = None,
                 n_shots: int = 3):
        self.model = model
        self.backend = backend
        self.system_prompt = SYSTEM_PROMPT
        self.few_shot_examples = self._build_few_shot(few_shot_df, n_shots)
        self._client = self._init_client()

    def _init_client(self):
        if self.backend == "ollama":
            import ollama
            return ollama
        elif self.backend == "openai_compatible":
            from openai import OpenAI
            return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

    def _build_few_shot(self, df, n_shots):
        """Sample n_shots labeled examples from English dataset per label."""
        if df is None:
            return []
        examples = []
        labeled = df[df['label'].notna()].copy()
        for label in PROPAGANDA_LABELS[:5]:  # top labels only
            subset = labeled[labeled['label'].str.contains(label, na=False)]
            for _, row in subset.sample(min(n_shots, len(subset))).iterrows():
                examples.append({
                    "text": row['text'],
                    "labels": row['label'].split(','),
                })
        return examples

    def _build_messages(self, text: str) -> list[dict]:
        messages = [{"role": "system", "content": self.system_prompt}]
        # Add few-shot examples
        for ex in self.few_shot_examples[:5]:
            messages.append({"role": "user", "content": ex["text"]})
            messages.append({"role": "assistant", "content":
                f'{{"labels": {json.dumps(ex["labels"])}, "justification": "example"}}'})
        messages.append({"role": "user", "content": text})
        return messages

    def label(self, text: str) -> dict:
        messages = self._build_messages(text)
        if self.backend == "ollama":
            response = self._client.chat(
                model=self.model,
                messages=messages,
                format="json"  # forces JSON output in ollama
            )
            return json.loads(response['message']['content'])

    def label_dataframe(self, df: pd.DataFrame, text_col: str,
                        id_col: Optional[str] = None) -> pd.DataFrame:
        results = []
        for _, row in df.iterrows():
            out = self.label(row[text_col])
            result = {
                "labels": ",".join(out.get("labels", [])),
                "justification": out.get("justification", ""),
                "text": row[text_col],
            }
            if id_col:
                result[id_col] = row[id_col]
            results.append(result)
        return pd.DataFrame(results)


In [28]:
sample_df = portuguese_df.head(15)
pt_sentences_sample = sentencize_df(sample_df, text_col="fake")
print(f"Total de sentenças: {len(pt_sentences_sample)}")

Total de sentenças: 97


In [30]:
def is_valid_sentence(text: str, min_words: int = 5) -> bool:
    words = text.split()
    return len(words) >= min_words

pt_sentences_sample_clean = pt_sentences_sample[
    pt_sentences_sample["sentence"].apply(is_valid_sentence)
].reset_index(drop=True)

print(f"Antes: {len(pt_sentences_sample)}")
print(f"Depois: {len(pt_sentences_sample_clean)}")

Antes: 97
Depois: 73


In [31]:
pt_sentences_sample_clean

,article_idx,sentence_idx,sentence
0,0,2,fazuele isso foi só um aperitivo do que tá por...
1,0,3,", 172, sexta feira de carnaval, em olinda os p..."
2,0,4,o brasil está entregue à criminalidade com a p...
3,1,0,"carro alegórico da escola de samba grande rio,..."
4,1,1,veja o vídeo é cada abominação nesse carnaval ...
...,...,...,...
68,13,4,"?, alberto fernández atual presidente da argen..."
69,14,1,"o ceo presidente da empresa smartmatic, que fa..."
70,14,2,"mais uma evidência de fraude, porém agora docu..."
71,14,3,", boa notícia preso nos eua pela interpol, o p..."


In [39]:
labeler = LLMLabeler()

In [42]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

def label_row(row):
    out = labeler.label(row["sentence"])
    return {
        "article_idx": row["article_idx"],
        "sentence_idx": row["sentence_idx"],
        "sentence": row["sentence"],
        "labels": ",".join(out.get("labels", [])),
        "justification": out.get("justification", ""),
    }

start = time.time()
results = [None] * len(pt_sentences_sample_clean)
rows = [row.to_dict() for _, row in pt_sentences_sample_clean.iterrows()]

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(label_row, row): i for i, row in enumerate(rows)}
    for future in tqdm(as_completed(futures), total=len(futures)):
        i = futures[future]
        results[i] = future.result()

elapsed = time.time() - start
print(f"Tempo total: {elapsed/60:.1f} minutos")
print(f"Por sentença: {elapsed/len(pt_sentences_sample_clean):.1f}s")

result_df = pd.DataFrame(results)
result_df.to_csv("labeled_pt_sample.csv", index=False)
result_df

100%|██████████| 73/73 [22:38<00:00, 18.61s/it] 


Tempo total: 22.6 minutos
Por sentença: 18.6s


,article_idx,sentence_idx,sentence,labels,justification
0,0,2,fazuele isso foi só um aperitivo do que tá por...,"Loaded_Language,Exaggeration-Minimisation,Flag...",The phrase 'apaetivo' uses loaded language to ...
1,0,3,", 172, sexta feira de carnaval, em olinda os p...","Loaded_Language,Name_Calling-Labeling,Exaggera...",The text uses strong emotional language like '...
2,0,4,o brasil está entregue à criminalidade com a p...,"Loaded_Language,Appeal_to_Fear-Prejudice",The phrase uses loaded language with terms lik...
3,1,0,"carro alegórico da escola de samba grande rio,...","Exaggeration-Minimisation,Appeal_to_Fear-Preju...",The text uses exaggeration by mentioning a car...
4,1,1,veja o vídeo é cada abominação nesse carnaval ...,"Loaded_Language,Appeal_to_Fear-Prejudice",The text uses 'abominação' which is a strongly...
...,...,...,...,...,...
68,13,4,"?, alberto fernández atual presidente da argen...","Name_Calling-Labeling,Appeal_to_Fear-Prejudice...",The text employs 'Name_Calling-Labeling' by re...
69,14,1,"o ceo presidente da empresa smartmatic, que fa...","Loaded_Language,Exaggeration-Minimisation",The text uses 'fabrica as urnas' (manufactures...
70,14,2,"mais uma evidência de fraude, porém agora docu...","Loaded_Language,Exaggeration-Minimisation",A expressão 'mais uma evidência' sugere que a ...
71,14,3,", boa notícia preso nos eua pela interpol, o p...","Loaded_Language,Appeal_to_Fear-Prejudice",The text uses loaded language by referring to ...
